# AIMLC ZG521 — Conversational AI · Group Assignment 1
## Problem Statement 2 — Study of Embedding Models and Approximate Nearest Neighbor Search: Semantic Quality vs Search Efficiency

**Group 129** · Total: 10 Marks · Deadline: 28 Aug 2026

## Student Details

| Name | BITS ID | Email |
|---|---|---|
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |

## Contribution by Each Student

| Member | Task(s) | Section(s) done |
|---|---|---|
| *TODO — name* | T1, T2, T8, report assembly | |
| *TODO — name* | T3, T4 | |
| *TODO — name* | T5 | |
| *TODO — name* | T6, T7 | |


## Problem Statement

Study of embedding models and approximate nearest neighbor (ANN) search, comparing **semantic quality vs search efficiency**:
- **Module 1** — Dataset and embedding preparation (1 mark)
- **Module 2** — Similarity metrics and exact retrieval (3 marks)
- **Module 3** — ANN search experiment: HNSW vs IVF (3 marks)
- **Module 4** — Embedding quality analysis and final recommendation (3 marks)


## Tools and Libraries Used

- **Python 3.10**
- `datasets` (Hugging Face) — loading the BEIR-format retrieval dataset
- `pandas`, `numpy` — data handling
- `sentence-transformers` — encoder embedding models (Task 2 onward)
- `faiss-cpu` — HNSW / IVF ANN indexes (Task 5 onward)
- `matplotlib` — plots (Task 6 onward)

Install all of the above with: `pip install -r requirements.txt` (see `ass-1/requirements.txt`).


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: set up the environment so downloads work behind a
# corporate proxy. It (0a) trusts the OS certificate store, (0b) routes HTTPS
# through the same proxy pip uses, then imports libraries, fixes the Group-129
# seed, and defines resolve_model() for local/offline model use.
# ----------------------------------------------------------------------
import os

# (0a) Trust the OS/corporate certificate store for HTTPS in this kernel.
#      One-time install:  pip install truststore
try:
    import truststore
    truststore.inject_into_ssl()
    print("[info] truststore active -- using the OS certificate store for HTTPS.")
except ModuleNotFoundError:
    print("[info] truststore not installed. Run:  pip install truststore  and re-run.")
except Exception as e:
    print(f"[info] truststore present but could not activate ({e!r}); continuing.")

# (0b) Route HTTP/HTTPS through the corporate proxy (the same one pip uses), so
#      Hugging Face downloads can reach the internet. Auto-detected from your
#      Windows/OS proxy settings; if that finds nothing, set PROXY_URL by hand.
#      Find your proxy with:  pip config list   or   netsh winhttp show proxy
#      Example:  PROXY_URL = "http://proxy.mycompany.com:8080"
PROXY_URL = r""
import urllib.request
_detected = urllib.request.getproxies()  # reads OS internet settings + *_PROXY env vars
_proxy = PROXY_URL or _detected.get("https") or _detected.get("http")
if _proxy:
    os.environ["HTTP_PROXY"] = _proxy
    os.environ["HTTPS_PROXY"] = _proxy
    # Some libraries read the lowercase names; set both.
    os.environ["http_proxy"] = _proxy
    os.environ["https_proxy"] = _proxy
    print(f"[info] Routing HTTPS through proxy: {_proxy}")
else:
    print("[info] No proxy auto-detected. If downloads fail with an SSL/handshake error,")
    print("       set PROXY_URL at the top of this cell (find it via 'pip config list').")

import sys, time, json, random
import numpy as np
import pandas as pd

random.seed(129)   # Group 129 -- fixed seed for reproducibility
np.random.seed(129)


def resolve_model(model_name):
    """
    If a local copy of the model exists under a models/<folder> directory,
    return that path (loads offline); otherwise return the Hugging Face id
    (which triggers a download). Searches the cwd and its parents plus the full
    in-repo path, so it works regardless of where the kernel was started.
    """
    local_name = model_name.split("/")[-1]
    bases = [
        os.path.join("models", local_name),
        os.path.join("2026-2027-Sem1", "AIMLCZG521-ConversationalAI",
                     "ass-1", "models", local_name),
    ]
    here = os.path.abspath(os.getcwd())
    for _ in range(10):
        bases.append(os.path.join(here, "models", local_name))
        bases.append(os.path.join(here, "ass-1", "models", local_name))
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent

    for base in bases:
        if os.path.isdir(base) and os.path.exists(os.path.join(base, "config.json")):
            print(f"[info] Using LOCAL model '{model_name}' from: {base} (no network needed)")
            return base
    print(f"[info] No local copy of '{model_name}' found -- will download from Hugging Face.")
    return model_name


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)


---
# Task 1 — Corpus and Query Dataset Preparation (0.5 Marks)

**Dataset chosen: `SciFact`** (BEIR benchmark; Thakur et al., 2021 / Wadden et al., 2020) — a publicly available dataset that ships a document corpus, a query set, and query-document relevance labels together, matching this task's requirement exactly. Corpus (~5,183 passages) and queries (~300) both clear the 1,000/50 minimums.


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: load the SciFact corpus/queries/qrels. It prefers the
# local copy under data/raw/ and only downloads from the Hugging Face Hub when
# those files cannot be found (the first-time case). Loading locally needs NO
# network, so it works on a machine behind an SSL-inspecting corporate proxy
# where the Hub download fails with an SSL handshake error.
#
# It prints two [diag] lines (the kernel working directory and where it found
# data/raw) so it is obvious which path was taken. If auto-detection ever
# fails, set ABS_PATH below to your data/raw folder and re-run.
# ----------------------------------------------------------------------
import os
import pandas as pd

# LAST-RESORT override: if the two [diag] lines below show "found at: None",
# in VS Code right-click the 'raw' folder -> "Copy Path", paste it between the
# quotes here, and re-run this cell. Example (Windows):
#   ABS_PATH = r"C:\\Users\\you\\MTech\\2026-2027-Sem1\\AIMLCZG521-ConversationalAI\\ass-1\\data\\raw"
ABS_PATH = r""

RAW_FILENAMES = ("corpus.jsonl", "queries.jsonl", "qrels.tsv")


def _find_data_raw_dir():
    """
    Return the first data/raw directory that contains all three raw files, or
    None. Checks ABS_PATH, the obvious relative path, the full in-repo path,
    and every data/raw (and ass-1/data/raw) from the current working directory
    upward -- so it works regardless of where the Jupyter kernel was started.
    """
    candidates = []
    if ABS_PATH:
        candidates.append(ABS_PATH)
    candidates += [
        os.path.join("data", "raw"),
        os.path.join("2026-2027-Sem1", "AIMLCZG521-ConversationalAI",
                     "ass-1", "data", "raw"),
    ]
    here = os.path.abspath(os.getcwd())
    for _ in range(10):
        candidates.append(os.path.join(here, "data", "raw"))
        candidates.append(os.path.join(here, "ass-1", "data", "raw"))
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent

    for base in candidates:
        if all(os.path.exists(os.path.join(base, f)) for f in RAW_FILENAMES):
            return base
    return None


# Resolve once so the "save" cell below can reuse the same directory.
DATA_DIR = _find_data_raw_dir() or os.path.join("data", "raw")


def _load_from_local(base):
    """Load the three tables from a local data/raw directory (no network)."""
    corpus_df = pd.read_json(os.path.join(base, "corpus.jsonl"), lines=True)
    queries_df = pd.read_json(os.path.join(base, "queries.jsonl"), lines=True)
    qrels_df = pd.read_csv(os.path.join(base, "qrels.tsv"), sep="\t")
    return corpus_df, queries_df, qrels_df


def _load_from_hub():
    """Download corpus/queries/qrels from the Hugging Face Hub (needs internet)."""
    from datasets import load_dataset
    corpus_ds = load_dataset("BeIR/scifact", "corpus", split="corpus")
    queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
    qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

    corpus_df = corpus_ds.to_pandas().rename(columns={"_id": "doc_id"})
    queries_df = queries_ds.to_pandas().rename(columns={"_id": "query_id"})
    qrels_df = qrels_ds.to_pandas()
    qrels_df.columns = ["query_id", "doc_id", "relevance"]
    return corpus_df, queries_df, qrels_df


def load_beir_scifact():
    """
    Load SciFact (corpus, queries, qrels), preferring the local data/raw copy
    and falling back to the Hugging Face Hub only when it is missing. Queries
    are filtered to those with a relevance judgment.
    """
    local_dir = _find_data_raw_dir()
    print(f"[diag] kernel working directory : {os.getcwd()}")
    print(f"[diag] local data/raw found at  : {local_dir}")

    if local_dir is not None:
        print("[info] Loading SciFact from local files (no network needed).")
        corpus_df, queries_df, qrels_df = _load_from_local(local_dir)
    else:
        print("[info] Local data/raw NOT found -- downloading from Hugging Face Hub.")
        print("       Behind a corporate proxy this can fail with an SSL handshake error.")
        print("       Fix: set ABS_PATH at the top of this cell to your data/raw folder.")
        corpus_df, queries_df, qrels_df = _load_from_hub()

    # IDs must be strings on both sides, or later merges/lookups silently fail
    corpus_df["doc_id"] = corpus_df["doc_id"].astype(str)
    queries_df["query_id"] = queries_df["query_id"].astype(str)
    qrels_df["query_id"] = qrels_df["query_id"].astype(str)
    qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)

    # Keep only queries with >=1 relevance judgment (standard BEIR practice)
    labelled_ids = set(qrels_df["query_id"].unique())
    before = len(queries_df)
    queries_df = queries_df[queries_df["query_id"].isin(labelled_ids)].reset_index(drop=True)
    if before - len(queries_df):
        print(f"[info] Dropped {before - len(queries_df)} queries with no relevance judgment "
              f"(kept {len(queries_df)}, all with >=1 qrel).")

    return corpus_df, queries_df, qrels_df


corpus_df, queries_df, qrels_df = load_beir_scifact()
print(f"corpus  : {len(corpus_df)} passages")
print(f"queries : {len(queries_df)} queries (all with >=1 relevance judgment)")
print(f"qrels   : {len(qrels_df)} relevance judgments")


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: check the loaded data against the assignment's stated
# minimums (>=1,000 passages, >=50 queries, every query labelled) and stop
# the notebook immediately (via assert) if either minimum isn't met.
# ----------------------------------------------------------------------
MIN_CORPUS, MIN_QUERIES = 1000, 50

meets_corpus_min = len(corpus_df) >= MIN_CORPUS
meets_query_min = len(queries_df) >= MIN_QUERIES

print(f"Corpus  >= {MIN_CORPUS}: {meets_corpus_min}  ({len(corpus_df)} passages)")
print(f"Queries >= {MIN_QUERIES}: {meets_query_min}  ({len(queries_df)} queries)")
print(f"Every query has >=1 relevance judgment: "
      f"{queries_df['query_id'].isin(qrels_df['query_id']).all()}")

assert meets_corpus_min, "Corpus below the 1,000-passage minimum"
assert meets_query_min, "Query set below the 50-query minimum"
print("\n[OK] SciFact data clears both minimums.")


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: print one example row from each of the three tables
# (corpus, queries, qrels) so the schema is visible before it's used later.
# ----------------------------------------------------------------------
print("=== Sample corpus passage ===")
print(corpus_df.iloc[0].to_dict())

print("\n=== Sample query ===")
print(queries_df.iloc[0].to_dict())

print("\n=== Sample relevance judgments (qrels) ===")
print(qrels_df.head())


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: write corpus/queries/qrels to the resolved data/raw
# directory (DATA_DIR, set in the load cell above) so Tasks 2+ can load the
# exact same fixed dataset without re-downloading or re-running this cell.
# ----------------------------------------------------------------------
import os

# Reuse the directory resolved in the load cell; default to ./data/raw if this
# cell is ever run on its own.
DATA_DIR = globals().get("DATA_DIR", os.path.join("data", "raw"))
os.makedirs(DATA_DIR, exist_ok=True)

corpus_df.to_json(os.path.join(DATA_DIR, "corpus.jsonl"), orient="records", lines=True)
queries_df.to_json(os.path.join(DATA_DIR, "queries.jsonl"), orient="records", lines=True)
qrels_df.to_csv(os.path.join(DATA_DIR, "qrels.tsv"), sep="\t", index=False)

print(f"Saved to {DATA_DIR}/: corpus.jsonl, queries.jsonl, qrels.tsv")


### Dataset Details and Source

- **Name:** SciFact (BEIR benchmark; biomedical / scientific-claim verification)
- **Citation:** Wadden et al., *Fact or Fiction: Verifying Scientific Claims*, EMNLP 2020; redistributed by Thakur et al., *BEIR*, NeurIPS 2021
- **Source:** https://huggingface.co/datasets/BeIR/scifact (corpus + queries), https://huggingface.co/datasets/BeIR/scifact-qrels (relevance judgments)
- **Size:** 5,183 corpus passages, 300 queries (each with ≥1 relevance judgment), binary relevance
- **Format:** corpus = `{doc_id, title, text}`; queries = `{query_id, text}`; qrels = `{query_id, doc_id, relevance}`

### Explanation of the Logic Used

`load_beir_scifact()` downloads corpus/queries/qrels from Hugging Face and normalizes IDs to a common schema. Queries are filtered to only those with a qrel — the raw split ships 1,109 queries but just 300 have a relevance judgment, so filtering makes "relevance information for each query" literally true.

### Justification for the Chosen Approach

The assignment explicitly permits "an existing dataset containing query-document relevance labels" — BEIR datasets bundle exactly that, avoiding hand-built (and inconsistent) relevance judgments. SciFact's claim-verification domain gives clean, unambiguous labels, useful later for Task 3/7.

### Inference

Corpus and queries clear the assignment minimums by ~5x and ~6x respectively, leaving room to subsample later if needed. The 809 dropped queries simply have no qrel in this split — expected, not a data quality issue.

### Limitations Observed

- Relevance is binary, not graded — gives Task 3's metric comparison less ranking nuance to work with.
- Only 283 of 5,183 documents are ever relevant to any query — a real "needle in haystack" ratio to keep in mind when reading Recall@5 later.

### Possible Improvements

- Cross-check against a second BEIR dataset (e.g. NFCorpus) as a robustness check on Task 7.
- If index-building is slow at full scale, subsample while keeping every relevant document per query.

### References

- Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS Datasets & Benchmarks.
- Wadden, D. et al. (2020). *Fact or Fiction: Verifying Scientific Claims.* EMNLP.


---
# Task 2 — Embedding Generation and Pooling (0.5 Marks)

**Two encoder models, chosen for contrasting profiles:**

1. `distilbert-base-uncased` (Sanh et al., 2019) — general-purpose distilled BERT, **mean pooling**, not fine-tuned for retrieval.
2. `BAAI/bge-large-en-v1.5` (Xiao et al., 2023) — trained specifically for retrieval via contrastive fine-tuning, **[CLS]-token pooling**.

## Why encoder models are appropriate

Encoder-only transformers use bidirectional self-attention — every token's representation draws on *both* directions of context — and pooling those representations yields one fixed-length vector summarizing the whole input's meaning. That's exactly what semantic retrieval needs: query and document mapped into a shared space where "similar meaning" becomes "small distance." Decoder-only models, by contrast, are causal (one-directional) and have no single hidden state that naturally summarizes a full sequence, making them a worse fit for embedding generation.


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: for each of the two models, record documented facts
# (dimension, pooling, max length), then produce embeddings for BOTH the
# corpus documents and the queries. Embeddings are SAVED to data/results/ and
# REUSED on later runs -- so once they exist (e.g. generated on a networked
# machine and committed), this cell and Tasks 3+ run with no model and no
# network (works on the office laptop offline). If a model must run but can't
# (no local copy / download blocked), it reports "N/A (not run here)".
# ----------------------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd

# Documented facts about each model (from official model cards / papers).
MODEL_SPECS = {
    "distilbert-base-uncased": {"pooling": "mean pooling", "dim": 768, "max_input_length": 512},
    "BAAI/bge-large-en-v1.5": {"pooling": "[CLS] token pooling", "dim": 1024, "max_input_length": 512},
}

os.makedirs("data/results", exist_ok=True)

corpus_texts = corpus_df["text"].tolist()
query_texts = queries_df["text"].tolist()

# Persist corpus-embedding times so the reported timing survives a reload.
TIMINGS_PATH = "data/results/task2_timings.json"
timings = {}
if os.path.exists(TIMINGS_PATH):
    try:
        timings = json.load(open(TIMINGS_PATH))
    except Exception:
        timings = {}

comparison_rows = []
for model_name, spec in MODEL_SPECS.items():
    print(f"=== {model_name} ===")
    safe = model_name.replace("/", "__")
    doc_path = f"data/results/embeddings_{safe}.npy"
    qry_path = f"data/results/query_embeddings_{safe}.npy"
    elapsed = None

    if os.path.exists(doc_path) and os.path.exists(qry_path):
        # Reuse previously saved embeddings -- no model, no network needed.
        doc_emb = np.load(doc_path)
        qry_emb = np.load(qry_path)
        elapsed = timings.get(model_name)
        print(f"  Loaded SAVED embeddings (no model needed): "
              f"docs={doc_emb.shape}, queries={qry_emb.shape}")
    else:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer(resolve_model(model_name))
            t0 = time.time()
            doc_emb = model.encode(corpus_texts, show_progress_bar=False)
            elapsed = time.time() - t0            # corpus embedding time = the reported metric
            qry_emb = model.encode(query_texts, show_progress_bar=False)
            np.save(doc_path, doc_emb)
            np.save(qry_path, qry_emb)
            timings[model_name] = round(elapsed, 2)
            json.dump(timings, open(TIMINGS_PATH, "w"), indent=2)
            print(f"  Generated + SAVED embeddings: docs={doc_emb.shape}, "
                  f"queries={qry_emb.shape}, corpus time={elapsed:.2f}s")
        except Exception as e:
            print(f"  [warning] Could not run this model here: {e!r}")
            print(f"  Fix: run cell 1 (proxy/truststore) then re-run; or place a local copy "
                  f"under models/{model_name.split('/')[-1]}/; or generate on a networked "
                  f"machine and commit data/results/*.npy.")

    comparison_rows.append({
        "Model name": model_name,
        "Embedding dimension": spec["dim"],
        "Max/typical input length (tokens)": spec["max_input_length"],
        "Pooling strategy": spec["pooling"],
        "Approx. time to embed corpus (s)": round(elapsed, 2) if elapsed is not None else "N/A (not run here)",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("data/results/task2_model_comparison.csv", index=False)
comparison_df


### Explanation of the Logic Used

Model name, dimension, max input length, and pooling strategy are documented facts from each model's official card (`MODEL_SPECS`) — no execution needed. Only the embeddings and timing require real execution, so the loop attempts that per model and reports `"N/A (not run here)"` on failure instead of fabricating a number.

### Justification for the Chosen Models

DistilBERT and BGE-large differ on exactly the axis this problem statement asks about: DistilBERT is smaller/faster and general-purpose, BGE-large is larger/slower and purpose-built for retrieval — and they use different pooling strategies (mean vs `[CLS]`), which is itself a required "for each model, document..." item. This gives Task 7 a real, explainable axis of disagreement rather than two near-identical models.

### Inference

BGE-large's larger dimension (1024 vs 768) and contrastive/RetroMAE training are expected to separate semantically related and unrelated passages more cleanly than DistilBERT — the concrete evidence for this is Task 7's side-by-side comparison, not asserted here.

### Limitations Observed

`sentence-transformers` isn't available in this authoring sandbox (no internet access to fetch model weights), so the timing column shows `"N/A (not run here)"`. Re-run this cell on the remote system for real embeddings and timing.

### Possible Improvements

- Batch the encoding call and report GPU vs CPU timing separately once run for real.
- Add a third, mid-sized model (e.g. `bge-base-en-v1.5`) to see if the quality/speed trade-off is smooth or has a knee.

### References

- Sanh, V. et al. (2019). *DistilBERT.* arXiv:1910.01108.
- Xiao, S. et al. (2023). *C-Pack* (BGE model family). arXiv:2309.07597.


---
# Task 3 — Similarity Metric Comparison (1.5 Marks)

**Model used: `BAAI/bge-large-en-v1.5`** — the retrieval-purpose-built model from Task 2, so the metric comparison reflects an embedding space actually optimized for semantic search (DistilBERT's general-purpose space would make this a less meaningful comparison).

**Pair selection (25 pairs, ≥20 required):** 5 randomly sampled queries × 5 candidate documents each (the true relevant document from `qrels`, plus 4 random distractors) — enough per query to compare *rankings*, not just isolated scores.


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: implement the three similarity/distance metrics
# directly from their mathematical definitions (not by calling an opaque
# library function), so the formulas are visible and auditable.
# ----------------------------------------------------------------------
import numpy as np

def cosine_similarity(a, b):
    """cos(theta) = (a . b) / (||a|| * ||b||)"""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def dot_product(a, b):
    """a . b = sum_i a_i * b_i"""
    return float(np.dot(a, b))

def l2_distance(a, b):
    """||a - b||_2 = sqrt(sum_i (a_i - b_i)^2)"""
    return float(np.linalg.norm(a - b))


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: build the 25 query-document pairs used for Task 3.
# For each of 5 randomly sampled queries, pair it with its one true relevant
# document (from qrels) plus 4 random "distractor" documents -- so each
# query has a small ranked set of candidates to compare metrics across.
# ----------------------------------------------------------------------
import random
random.seed(129)

MODEL_FOR_TASK3 = "BAAI/bge-large-en-v1.5"
N_QUERIES, N_DISTRACTORS = 5, 4

sample_queries = queries_df.sample(n=N_QUERIES, random_state=129).reset_index(drop=True)
all_doc_ids = corpus_df["doc_id"].tolist()

pairs = []
for _, q in sample_queries.iterrows():
    qid, qtext = q["query_id"], q["text"]

    # The one document SciFact's qrels marks as relevant to this query
    rel_docs = qrels_df[qrels_df["query_id"] == qid]["doc_id"].tolist()
    true_doc_id = rel_docs[0] if rel_docs else None

    # 4 random documents NOT known to be relevant, to give each query a
    # small ranked candidate set (relevant + distractors) instead of just
    # one isolated score
    distractor_pool = [d for d in all_doc_ids if d != true_doc_id]
    distractors = random.sample(distractor_pool, N_DISTRACTORS)

    doc_ids_for_query = ([true_doc_id] if true_doc_id else []) + distractors
    for did in doc_ids_for_query:
        text = corpus_df.loc[corpus_df["doc_id"] == did, "text"].values[0]
        pairs.append({"query_id": qid, "query_text": qtext, "doc_id": did,
                      "doc_text": text, "is_relevant": did == true_doc_id})

pairs_df = pd.DataFrame(pairs)
print(f"Selected {len(pairs_df)} query-document pairs across {pairs_df['query_id'].nunique()} "
      f"queries (>= 20 required).")


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: get an embedding for each pair's query and document,
# then compute cosine similarity, dot product, and L2 distance for every pair.
# It REUSES the Task 2 saved embeddings for MODEL_FOR_TASK3 when they exist
# (no model / no network); otherwise it loads the model and embeds the pairs.
# ----------------------------------------------------------------------
import os
import numpy as np
import pandas as pd

_safe3 = MODEL_FOR_TASK3.replace("/", "__")
_doc_path = f"data/results/embeddings_{_safe3}.npy"
_qry_path = f"data/results/query_embeddings_{_safe3}.npy"

metrics_df = None
q_emb = d_emb = None


def _compute_metrics(q_emb, d_emb):
    rows = []
    for i in range(len(pairs_df)):
        a, b = q_emb[i], d_emb[i]
        rows.append({
            "query_id": pairs_df.loc[i, "query_id"],
            "doc_id": pairs_df.loc[i, "doc_id"],
            "is_relevant": pairs_df.loc[i, "is_relevant"],
            "cosine": cosine_similarity(a, b),
            "dot_product": dot_product(a, b),
            "l2_distance": l2_distance(a, b),
        })
    return pd.DataFrame(rows)


if os.path.exists(_doc_path) and os.path.exists(_qry_path):
    # Reuse Task 2 embeddings. They were saved in corpus_df / queries_df row
    # order, so map each pair's id back to its row to pull the right vector.
    doc_emb_all = np.load(_doc_path)
    qry_emb_all = np.load(_qry_path)
    doc_idx = {d: i for i, d in enumerate(corpus_df["doc_id"].tolist())}
    qry_idx = {q: i for i, q in enumerate(queries_df["query_id"].tolist())}
    q_emb = np.stack([qry_emb_all[qry_idx[pairs_df.loc[i, "query_id"]]] for i in range(len(pairs_df))])
    d_emb = np.stack([doc_emb_all[doc_idx[pairs_df.loc[i, "doc_id"]]] for i in range(len(pairs_df))])
    metrics_df = _compute_metrics(q_emb, d_emb)
    metrics_df.to_csv("data/results/task3_similarity_metrics.csv", index=False)
    print(f"Reused SAVED {MODEL_FOR_TASK3} embeddings (no model needed) for {len(metrics_df)} pairs.")
else:
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(resolve_model(MODEL_FOR_TASK3))
        q_emb = model.encode(pairs_df["query_text"].tolist(), show_progress_bar=False)
        d_emb = model.encode(pairs_df["doc_text"].tolist(), show_progress_bar=False)
        metrics_df = _compute_metrics(q_emb, d_emb)
        metrics_df.to_csv("data/results/task3_similarity_metrics.csv", index=False)
        print(f"Computed cosine / dot-product / L2 for {len(metrics_df)} pairs using {MODEL_FOR_TASK3}.")
    except Exception as e:
        metrics_df = None
        q_emb = d_emb = None
        print(f"[warning] Could not run {MODEL_FOR_TASK3} here: {e!r}")
        print("Run cell 1 first, or run Task 2 to create saved embeddings, then re-run.")


In [ ]:
# ----------------------------------------------------------------------
# What this cell does: two checks required by Task 3.
#   1) Ranking comparison -- for each query, does sorting its 5 candidate
#      docs by cosine give the same order as sorting by dot product / L2?
#   2) Normalization effect -- re-normalize the same embeddings to unit
#      length and confirm the textbook identities: dot(a,b) == cosine(a,b)
#      and ||a-b||^2 == 2 - 2*cosine(a,b) once vectors have length 1.
# ----------------------------------------------------------------------
if metrics_df is not None:
    print("=== Ranking comparison per query (raw, unnormalized embeddings) ===")
    for qid, group in metrics_df.groupby("query_id"):
        cos_rank = group.sort_values("cosine", ascending=False)["doc_id"].tolist()
        dot_rank = group.sort_values("dot_product", ascending=False)["doc_id"].tolist()
        l2_rank = group.sort_values("l2_distance", ascending=True)["doc_id"].tolist()  # smaller distance = closer
        print(f"query {qid}: cosine==dot ranking? {cos_rank == dot_rank}  |  "
              f"cosine==L2 ranking? {cos_rank == l2_rank}")

    print("\n=== Normalization effect: recompute dot/L2 on unit-normalized embeddings ===")
    def normalize(v):
        return v / np.linalg.norm(v)

    norm_rows = []
    for i in range(len(pairs_df)):
        a, b = normalize(q_emb[i]), normalize(d_emb[i])
        norm_rows.append({
            "query_id": pairs_df.loc[i, "query_id"],
            "doc_id": pairs_df.loc[i, "doc_id"],
            "dot_normalized": dot_product(a, b),
            "l2_normalized": l2_distance(a, b),
        })
    norm_df = pd.DataFrame(norm_rows).merge(
        metrics_df[["query_id", "doc_id", "cosine"]], on=["query_id", "doc_id"])
    # These two columns should both end up all-True -- that's the mathematical
    # identity for unit-normalized vectors, verified here on real numbers
    norm_df["dot_matches_cosine"] = np.isclose(norm_df["dot_normalized"], norm_df["cosine"], atol=1e-4)
    norm_df["l2sq_matches_2_minus_2cos"] = np.isclose(
        norm_df["l2_normalized"] ** 2, 2 - 2 * norm_df["cosine"], atol=1e-4)
    print(norm_df[["query_id", "doc_id", "dot_matches_cosine", "l2sq_matches_2_minus_2cos"]].to_string(index=False))
else:
    print("Skipped -- needs metrics_df from the cell above (requires live model access).")


### Explanation of the Logic Used

All three metrics are implemented directly from their mathematical definitions (see docstrings above), not called as opaque library functions. The 25 pairs let rankings be compared *per query* (5 docs ranked 3 different ways), and the normalization cell re-embeds nothing — it just unit-normalizes the same vectors and recomputes, isolating normalization as the only variable.

### Justification for the Chosen Approach

Comparing rankings (not just raw scores) is what actually answers "does the metric matter" — two metrics can disagree on absolute values yet agree on which document ranks first, which is what retrieval cares about. Testing the normalized case directly demonstrates the identities `dot(â,b̂) = cos(a,b)` and `‖â-b̂‖² = 2 - 2·cos(a,b)` for unit vectors â, b̂ — not asserted from theory, but shown to hold (or not) on the actual embeddings.

### Inference

*(Fill in after running on the remote system — expected pattern: on raw embeddings, dot-product ranking can diverge from cosine's whenever vector norms vary across documents; once normalized, dot product and cosine rankings become identical by the mathematical identity above, and L2 ranking (ascending distance) matches cosine ranking (descending similarity) exactly.)*

### Limitations Observed

Only 5 queries were sampled (25 pairs total) — enough to satisfy the ≥20-pair minimum and show the pattern, but too few to claim it holds for the full 300-query set without re-running at larger scale.

### Possible Improvements

- Re-run across all 300 queries once compute allows, to confirm the ranking-identity pattern holds at scale, not just for 5 samples.
- Repeat with DistilBERT's embeddings for comparison — an embedding space *not* trained for retrieval may show the metric choice mattering more.

### References

- Formulas per standard linear algebra definitions (dot product, Euclidean norm, cosine of the angle between vectors).


---
# Task 4 — Exact kNN Baseline (1.5 Marks)
*Not started.*


---
# Task 5 — HNSW vs IVF (2 Marks)
*Not started.*


---
# Task 6 — ANN Trade-off Analysis (1 Mark)
*Not started.*


---
# Task 7 — Qualitative Retrieval Analysis (2 Marks)
*Not started.*


---
# Task 8 — Final Recommendation (1 Mark)
*Not started.*


---
# Final Conclusion
*To be written once Tasks 2–8 are complete — must cover key observations, strengths, limitations, and possible future improvements.*


# References
*Consolidated reference list — add each task's citations here as they're completed.*

- Thakur, N. et al. (2021). BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models. NeurIPS.
- Wadden, D. et al. (2020). Fact or Fiction: Verifying Scientific Claims. EMNLP.
